In [1]:
import os
import sys

sys.path.append("/home/koledjoz/Ex2VecExtended")

In [2]:
import torch
import pandas as pd
from tqdm import tqdm
import numpy as np

from src.models.optimized.original import Ex2VecOriginalFast

In [3]:
df = pd.read_parquet('../../../../sorted_data.parquet')


In [4]:
max_history = 3500
user_count = df['user_id'].max()
item_count = df['track_id'].max()

In [5]:
time_history = np.zeros((user_count+1, max_history), dtype=int)
item_history = np.zeros((user_count+1, max_history), dtype=int)

for user in tqdm(df['user_id'].unique()):
    tmp = df[df['user_id'] == user]
    item_history[user, :len(tmp)] = tmp['track_id'].to_numpy()
    time_history[user, :len(tmp)] = tmp['ts'].to_numpy()

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 13209/13209 [11:25<00:00, 19.28it/s]


In [6]:
checkpoint_path = "../train_notebooks/original.pt"
checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=False)

In [7]:
from torch.utils.data import Dataset

class DatasetWrapper(torch.utils.data.Dataset):
    def __init__(self, predict_dict, start_time, end_time, n_steps):
        self.times = []
        self.users = []
        self.items = []

        for u, vals in predict_dict.items():
            for i in vals:
                for t in np.linspace(start_time, end_time, n_steps):
                    self.times.append(int(t))
                    self.users.append(int(u))
                    self.items.append(i)
        

    def __len__(self):
        return len(self.times)

    def __getitem__(self, idx):
        return self.times[idx], self.users[idx], self.items[idx]

In [11]:
import json

with open('../../../../split_data/test/test_dict.json', 'r') as f:
    data_dict = json.load(f)

data = DatasetWrapper(data_dict, 1654041600, 1661990376, 200)

In [15]:
pairs = {(int(u), i) for u, items in data_dict.items() for i in items}
df = df[~df[['user_id','track_id']].apply(tuple, axis=1).isin(pairs)]

In [9]:
import torch
from torch.utils.data import DataLoader

# data = Ex2VecDataset(filtered_df['ts'].to_numpy(), filtered_df['user_id'].to_numpy(), filtered_df['track_id'].to_numpy())

batch_size = 2**14

# train_dataloader = DataLoader(data, batch_size=batch_size, shuffle=True)

device = 'cpu'
config = {'n_users': user_count, 'n_items': item_count, 'latent_d': 64}


model = Ex2VecOriginalFast(config)
model.load_state_dict(checkpoint["model_state_dict"], strict=False)
model.initialize_histories(torch.tensor(item_history).to(device), torch.tensor(time_history).to(device))
model.to(device)

NameError: name 'Ex2VecDataset' is not defined

In [ ]:
model

In [ ]:
top_k = 50
all_batches = []

prediction_cols = [f"pred_{i + 1}" for i in range(top_k)]

with torch.no_grad():
    model.eval()
    dataloader = DataLoader(data, batch_size=batch_size, shuffle=False)
    for step, batch in enumerate(tqdm(dataloader)):
        model_result = model(batch[0].to(device), batch[1].to(device)).cpu().numpy()

        user_id = batch[1].cpu().numpy()
        item_id = batch[2].cpu().numpy()
        ts = batch[0].cpu().numpy()

        idx = np.argsort(-model_result, axis=1)
        predict_items = idx[:, :top_k]

        u = user_id.reshape(-1)[:, None]  # -> (B, 1)
        i = item_id.reshape(-1)[:, None]  # -> (B, 1)
        t = ts.reshape(-1)[:, None]  # -> (B, 1)
        c = predict_items

        data = np.concatenate([u, i, t, c], axis=1)

        batch_df = pd.DataFrame(data, columns=["userId", "trackId", "ts"] + prediction_cols)
        all_batches.append(batch_df)


    df_preds = pd.concat(all_batches, ignore_index=True)
    # df_preds.to_csv(output_path, index=False)


In [ ]:
df_preds.to_csv('./original_output.csv', index=False)